In [44]:
import pandas as pd

import numpy as np

import matplotlib.pyplot as plt

import seaborn as sns

In [45]:
# This problem statement wants us to predict the Accident riks given the many signals such as the road curvature, etc

df = pd.read_csv("train.csv")

df_test = pd.read_csv("test.csv")

In [46]:
df.head(5)

,id,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
0,0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1,0.13
1,1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0,0.35
2,2,rural,4,0.63,70,dim,clear,False,True,morning,True,False,2,0.30
3,3,highway,4,0.07,35,dim,rainy,True,True,morning,False,False,1,0.21
4,4,rural,1,0.58,60,daylight,foggy,False,False,evening,True,False,1,0.56


In [47]:
df = df.drop(columns = 'id', axis = 1)

In [48]:
cat = []

for col in df.columns:
    cat.append(col)


In [49]:
df['road_type'].dtypes

dtype('O')

In [50]:
df.dtypes

road_type                  object
num_lanes                   int64
curvature                 float64
speed_limit                 int64
lighting                   object
weather                    object
road_signs_present           bool
public_road                  bool
time_of_day                object
holiday                      bool
school_season                bool
num_reported_accidents      int64
accident_risk             float64
dtype: object

In [51]:
df.head(5)

,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1,0.13
1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0,0.35
2,rural,4,0.63,70,dim,clear,False,True,morning,True,False,2,0.30
3,highway,4,0.07,35,dim,rainy,True,True,morning,False,False,1,0.21
4,rural,1,0.58,60,daylight,foggy,False,False,evening,True,False,1,0.56


In [52]:
df['road_type'].unique()
df['lighting'].unique()
df['time_of_day'].unique()

array(['afternoon', 'evening', 'morning'], dtype=object)

In [53]:
df.head(2)

,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1,0.13
1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0,0.35


In [54]:
from sklearn.model_selection import train_test_split
X = df.iloc[:, :-1]

y = df.iloc[:, -1]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [55]:
df.columns

Index(['road_type', 'num_lanes', 'curvature', 'speed_limit', 'lighting',
       'weather', 'road_signs_present', 'public_road', 'time_of_day',
       'holiday', 'school_season', 'num_reported_accidents', 'accident_risk'],
      dtype='object')

In [56]:
df.head(2)

,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1,0.13
1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0,0.35


In [57]:
# Ordinal Encoders - lighting, time_of_day

# Nominal Encoders - Weather, road_signs_present, public_road, school_season

from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
# We can simply first define the columns that we need for Ordinal or One Hot Encoding
nominal_encoding = ['road_type', 'weather', 'road_signs_present', 'public_road', 'school_season']
ordinal_columns = ['lighting', 'time_of_day']
ordinal_encoding_lighting = ['night', 'dim', 'daylight']
ordinal_encoding_time = ['morning', 'afternoon', 'evening']

# We will apply Standard Scaler for scaling all the numerical columns

numerical = ['num_lanes', 'curvature', 'speed_limit', 'num_reported_accidents']

preprocessor = ColumnTransformer(
    transformers = [
        ('onehotencoder', OneHotEncoder(drop = 'first', sparse_output = False), nominal_encoding),
        ('ordinal', OrdinalEncoder(categories = [ordinal_encoding_lighting, ordinal_encoding_time]), ordinal_columns),
         ('stdscaler', StandardScaler(), numerical)      
         ],
    remainder = 'passthrough'
)

X_train_transform = preprocessor.fit_transform(X_train)
X_validate_transform = preprocessor.transform(X_test)
X_test_transform = preprocessor.transform(df_test)







In [58]:
df.head(5)

,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1,0.13
1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0,0.35
2,rural,4,0.63,70,dim,clear,False,True,morning,True,False,2,0.30
3,highway,4,0.07,35,dim,rainy,True,True,morning,False,False,1,0.21
4,rural,1,0.58,60,daylight,foggy,False,False,evening,True,False,1,0.56


In [59]:
# Converting the dataframe back from a nparray to a dataframe

feature_names = preprocessor.get_feature_names_out()
feature_names1 = preprocessor.get_feature_names_out()
feature_names2 = preprocessor.get_feature_names_out()
X_train_df = pd.DataFrame(X_train_transform, columns = feature_names, index = X_train.index)

X_validate_df = pd.DataFrame(X_validate_transform, columns = feature_names1, index = X_test.index)

X_test_df = pd.DataFrame(X_test_transform, columns = feature_names2, index = df_test.index)

In [64]:
from sklearn.linear_model import LinearRegression


model = LinearRegression()

model_pred = model.fit(X_train_df, y_train)

y_pred1 = model_pred.predict(X_test_df)



In [65]:
submission = pd.read_csv("sample_submission.csv")

In [69]:
submission.head(2)

submission.shape

(172585, 2)

In [68]:
y_pred1.shape

(172585,)

In [71]:
submission['accident_risk'] = y_pred1

In [72]:
submission.to_csv("submission1_linear_regression.csv", index = False)